# 深度學習框架應用（大學部）｜主題 3 延伸：整合複習

這本筆記本配合投影片《主題 3 延伸：整合複習》，把主題 2、主題 3 學過的東西串成一圈：照片裝成一盒張量 → 一個神經元投票 → 損失函數扣分 → 跟著梯度下山 → 五步訓練迴圈。**練習一～五，每題 20%，合計 100%**。

| 練習 | 題目 | 投影片（練習頁） | 觀念在哪一頁 |
|---|---|---|---|
| 一 | 點心照片裝箱（張量與 batch） | 第 25 頁 | 第 7、15 頁 |
| 二 | 一位評審投一票（神經元與 ReLU） | 第 26 頁 | 第 8、9 頁 |
| 三 | 猜錯扣幾分（損失函數） | 第 27 頁 | 第 11 頁 |
| 四 | 蒙眼下山（梯度、學習率、鏈鎖律） | 第 28 頁 | 第 12、13 頁 |
| 五 | 小智的五步練習圈（訓練迴圈） | 第 29 頁 | 第 14、15 頁 |

**怎麼用**

1. 用私人 Google 帳號登入（不使用學校帳號），按上方的「複製到雲端硬碟」（或「檔案 › 在雲端硬碟中儲存複本」）另存一份自己的複本，之後都改自己那份。**沒有另存就直接寫，分頁一關就全部不見。** 存好後先把檔名改成 `深度學習_W03延伸_學號_姓名.ipynb`（左上角點檔名就能改）。
2. 先跑「環境準備」，再由上往下做。每一題只有一格標 ✏️ 的程式格：把每個 `___` 換成答案再執行，提示就寫在同一行後面的註解。還沒把 `___` 換掉就執行，會出現紅字錯誤（例如 TypeError、AttributeError），錯誤訊息裡箭頭 `---->` 指的那一行就是還沒填的空格，填好再跑一次就好。
3. 這份都是小張量，CPU 就跑得動，不必切 GPU。改壞了就重新開原本的連結，再存一份複本。
4. **每題做完就截一張整個螢幕的圖（截圖 ①～⑤）**，要看得到身份指紋：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間，以及這一題的輸出。**看不到身份指紋，那一題不計分。**

**繳交**：做完照最後一格「繳交前檢查」，把**分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤）**交到 Flipclass 作業區。

## 環境準備

五題都會用到下面這些套件，先跑這一格。

In [ ]:
# ===== 環境準備 =====
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
print('PyTorch：', torch.__version__, '｜CUDA 可用：', torch.cuda.is_available())

## 練習一：點心照片裝箱 (20%)

> 題目在投影片**第 25 頁**｜觀念在第 7、15 頁

**情境**：小智的點心店拍了 1000 張點心照片，每張是 32×32 的彩色照片，一次裝 16 張一盒（一個 batch）送去練習。

**這題在練**：說得出一盒照片的 shape 每個數字代表什麼，會數元素個數、會攤平，也會算一個 epoch（全部照片看過一輪）要搬幾盒

**步驟**

1. 在 `torch.rand(___, ___, ___, ___)` 填入四個數字，做出一盒 16 張 32×32 的彩色照片，印出 shape
2. 用 `.numel()` 數一盒裡共有幾個數字，再用 `.flatten()` 把每張照片攤平成一排（張數那一維要留著）
3. 用 `math.ceil`（無條件進位）算出 1000 張照片要搬幾盒，才算跑完一個 epoch

**要繳交**

- 截圖 ①：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ✏️ 練習一：點心照片裝箱
import math
x = torch.rand(___, ___, ___, ___)  # ✏️ 張數,通道,高,寬
print('一盒照片的 shape：', x.shape)
print('共有幾個數字：', x.___())    # ✏️ 數元素個數
flat = x.flatten(___)               # ✏️ 留下張數那一維
print('攤平後的 shape：', flat.shape)
boxes = math.ceil(___ / ___)        # ✏️ 總張數 / 每盒張數
print('一個 epoch 要搬幾盒：', boxes)

#### 📸 截圖 ①

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：四行結果（一盒的 shape、一盒共有幾個數字、攤平後的 shape、一個 epoch 要搬幾盒）

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. shape (16, 3, 32, 32) 的四個數字，各代表什麼？**

（在這裡寫）

**2. 為什麼 1000 張要搬 63 盒？最後一盒有幾張？**

（在這裡寫）

## 練習二：一位評審投一票 (20%)

> 題目在投影片**第 26 頁**｜觀念在第 8、9 頁

**情境**：小智請一位評審看三個線索，決定「這是不是巧克力餅乾」：圓不圓、甜不甜、有沒有巧克力。這一片的線索是 x = [1.0, 0.8, 0.0]（圓、很甜、沒有巧克力），評審的權重是 w = [0.5, 1.0, 2.0]，偏移 b = −1。

**這題在練**：親手算一個神經元的「乘起來再加起來」，再過 ReLU（Rectified Linear Unit），並證明 `nn.Linear` 算的就是同一件事

**步驟**

1. 先自己算 z：三個線索各乘上自己的權重、加起來、再加上 b；接著把 z 放進 ReLU
2. 建一個 `nn.Linear(3, 1)`，把權重和偏移設成同樣的值，丟 x 進去，結果要和第 1 步一樣
3. 把第三個線索改成 1.0（有巧克力），再算一次 z

**要繳交**

- 截圖 ②：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ✏️ 練習二：一位評審投一票
x = torch.tensor([1.0, 0.8, 0.0])   # 圓、甜、沒有巧克力
w = torch.tensor([0.5, 1.0, 2.0])   # 三個線索的權重
b = -1.0                            # 偏移 b
z = (___ * ___).sum() + ___         # ✏️ 乘起來再加起來
r = torch.relu(___)                 # ✏️ 把 z 放進 ReLU
print(f'自己算：z = {z:.4f}，ReLU 後 = {r:.4f}')
layer = nn.Linear(3, 1)             # 3 進 1 出
with torch.no_grad():               # 手動設值，不要記帳
    layer.weight[:] = w
    layer.bias[:] = b
print(f'nn.Linear 算：{layer(x).item():.4f}')
x2 = torch.tensor([1.0, 0.8, ___])  # ✏️ 有巧克力
print(f'有巧克力：z = {(w * x2).sum() + b:.4f}')

#### 📸 截圖 ②

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：三行結果（自己算的 z 和 ReLU 後的值、nn.Linear 算出的值、有巧克力時的 z）

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. b = −1 在這裡像什麼？**

（在這裡寫）

**2. 如果 z 算出來是負的（例如 z = −0.5），ReLU 之後會變多少？**

（在這裡寫）

## 練習三：猜錯扣幾分 (20%)

> 題目在投影片**第 27 頁**｜觀念在第 11 頁

**情境**

- **猜價錢（數字題）**：三樣點心真正的價錢是 [30, 50, 20]。小明猜 A = [32, 48, 21]，每個都差一點；小華猜 B = [30, 50, 35]，只錯一個，但錯很大。
- **猜種類（選擇題）**：三類是 0 餅乾、1 蛋糕、2 糖果，這一片的正確答案是 0（餅乾）。比較三種猜法：很有把握且猜對 [3, 0, 0]、很有把握但猜錯 [0, 3, 0]、完全亂猜 [0, 0, 0]。

**這題在練**：用 MSE（Mean Squared Error，均方誤差）、MAE（Mean Absolute Error，平均絕對誤差）和 CE（Cross Entropy，交叉熵）算出「猜錯扣幾分」，看懂不同的損失函數怎麼扣分

**步驟**

1. 用 `nn.MSELoss()` 和 `nn.L1Loss()`（MAE 在 PyTorch 裡的名字）算小明（A）、小華（B）各被扣幾分
2. 用 `nn.CrossEntropyLoss()` 算三種猜法的扣分；標籤放正確答案的類別編號，寫成一維張量 `torch.tensor([編號])`
3. 比一比：哪一種錯法被扣最多？MSE 和 MAE 對小華的扣法差在哪？

> **小提醒**：`CrossEntropyLoss` 吃的是 logits（模型最後算出來、還沒做 softmax 的分數），直接把分數丟進去就好，不要自己先做 softmax。

**要繳交**

- 截圖 ③：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ✏️ 練習三：猜錯扣幾分
y = torch.tensor([30., 50., 20.])      # 真正的價錢
A = torch.tensor([32., 48., 21.])      # 小明：每個差一點
B = torch.tensor([30., 50., 35.])      # 小華：只錯一個
mse, mae = nn.___(), nn.___()          # ✏️ 先 MSE，再 MAE
print(f'MSE：A = {mse(A, y):.3f}，B = {mse(B, y):.3f}')
print(f'MAE：A = {mae(A, y):.3f}，B = {mae(B, y):.3f}')
ce = nn.___()                          # ✏️ 交叉熵
label = torch.tensor([___])            # ✏️ 餅乾是第幾類？
for name, z in [('有把握、猜對', [[3., 0., 0.]]),
                ('有把握、猜錯', [[0., 3., 0.]]),
                ('完全亂猜', [[0., 0., 0.]])]:
    loss = ce(torch.tensor(z), label)  # 直接給 logits
    print(f'{name}：CE = {loss:.4f}')

#### 📸 截圖 ③

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：五行結果（MSE 和 MAE 各兩個分數、三種猜法的 CE）

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 小華（B）只錯一個，為什麼 MSE 扣得比小明（A）多很多？**

（在這裡寫）

**2. 猜價錢和猜種類，各該用哪一種損失函數？**

（在這裡寫）

## 練習四：蒙眼下山 (20%)

> 題目在投影片**第 28 頁**｜觀念在第 12、13 頁

**情境**：蒙著眼睛下山，看不到谷底在哪裡，只能用腳感覺「哪邊是下坡」（梯度），再決定一步要跨多大（學習率）。

**這題在練**：用 autograd 驗證鏈鎖律，再親眼看學習率太小、剛好、太大，走起來差多少

**步驟**

1. 鏈鎖律齒輪：w = 1，y = (2w + 1)²，用 autograd 反向傳播算出 dy/dw，印出 `w.grad`
2. 學習率：山谷是 L(w) = (w − 4)²，從 w = 0 出發，lr（learning rate，學習率）分別用 0.01、0.3、1.1 各走 10 步；每一步都是「反向傳播 → 往下坡走一步 → 梯度歸零」
3. 比一比三個 lr 最後各走到哪裡（谷底在 w = 4）

**要繳交**

- 截圖 ④：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ✏️ 練習四：蒙眼下山
w = torch.tensor(1.0, requires_grad=True)
y = (2 * w + 1) ** 2           # 兩個齒輪：先 2w+1，再平方
y.___()                        # ✏️ 反向傳播，算出梯度
print('鏈鎖律：dy/dw =', w.grad.item())
for lr in [___, ___, ___]:     # ✏️ 三個學習率
    w = torch.tensor(0.0, requires_grad=True)  # 從 0 出發
    for _ in range(10):        # 走 10 步
        L = (w - 4) ** 2       # 谷底在 w = 4
        L.backward()
        with torch.no_grad():  # 更新時不要記帳
            w -= ___ * w.grad  # ✏️ 往下坡走一步
        w.grad.___()           # ✏️ 舊梯度歸零
    print(f'lr = {lr}：10 步後 w = {w.item():.4f}')

#### 📸 截圖 ④

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：四行結果（鏈鎖律算出的 dy/dw、三個 lr 走 10 步後的 w）

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 手算 y = (2w + 1)² 在 w = 1 的 dy/dw（寫出鏈鎖律的過程），和程式印出的 w.grad 對答案。**

（在這裡寫）

**2. lr = 0.01、0.3、1.1 各像哪一種走路方式？為什麼每走一步，都要把 w.grad 歸零？**

（在這裡寫）

## 練習五：小智的五步練習圈 (20%)

> 題目在投影片**第 29 頁**｜觀念在第 14、15 頁

**情境**：每多一顆糖果貴 2 元，盒子 1 元，所以價錢 y = 2x + 1。我們不告訴模型 2 和 1，只給它 10 筆「糖果數 → 價錢」的資料，讓它自己學出來。

**這題在練**：把主題 2、主題 3 學過的東西串成訓練迴圈，每一盒資料都照五步走：猜 → 算扣分 → 梯度歸零 → 反向傳播 → 走一步

**步驟**

1. 先執行「準備資料」那一格：10 筆資料、每盒 4 筆、模型 `nn.Linear(1, 1)`、最佳化器 SGD（Stochastic Gradient Descent，隨機梯度下降）、損失函數 MSE
2. 在 ✏️ 格填好五步裡的空格，練 200 個 epoch，每個 epoch 記下平均 loss
3. 看印出來的 w、b 有沒有接近 2 和 1，loss 曲線有沒有一路往下

> 想重跑 ✏️ 格，要先重跑「準備資料」，模型才會回到亂猜的起點。

**要繳交**

- 截圖 ⑤：整個螢幕，看得到身份指紋和這一題的輸出
- 文字題：寫在下面的「✏️ 文字回答」

In [ ]:
# ===== 練習五：準備資料（直接執行，不用改）=====
# 想重跑下面的 ✏️ 格，要先重跑這一格，模型才會回到亂猜的起點
torch.manual_seed(0)                                     # 固定亂數，大家的結果才會一樣
x = torch.arange(10, dtype=torch.float32).unsqueeze(1)   # 0～9 顆糖果，shape (10, 1)
y = 2 * x + 1                                            # 價錢：每顆 2 元，再加盒子 1 元
dl = DataLoader(TensorDataset(x, y), batch_size=4, shuffle=True)  # 每盒 4 筆，每一輪重新洗牌
model = nn.Linear(1, 1)                                  # 模型：猜的價錢 = w * 糖果數 + b
opt = torch.optim.SGD(model.parameters(), lr=0.02)       # 最佳化器：SGD，學習率 0.02
loss_fn = nn.MSELoss()                                   # 損失函數：MSE
hist = []                                                # 記錄每個 epoch 的平均 loss
print('一個 epoch 有幾盒：', len(dl))
print(f'練習前（亂猜的）：w = {model.weight.item():.3f}，b = {model.bias.item():.3f}')

In [ ]:
# ✏️ 練習五：五步練習圈，練 200 個 epoch
for epoch in range(200):
    total = 0.0                   # 這一輪的 loss 加總
    for xb, yb in dl:             # 一次拿一盒
        pred = ___(xb)            # ✏️ ① 交給模型猜
        loss = loss_fn(pred, yb)  # ② 算扣幾分
        opt.___()                 # ✏️ ③ 舊梯度歸零
        loss.___()                # ✏️ ④ 反向傳播
        opt.___()                 # ✏️ ⑤ 往下坡走一步
        total += loss.item()
    hist.append(total / len(dl))  # 記下這一輪的平均
w, b = model.weight.item(), model.bias.item()
print(f'w = {w:.3f}，b = {b:.3f}，loss = {hist[-1]:.1e}')
plt.plot(hist); plt.yscale('log'); plt.show()

#### 📸 截圖 ⑤

跑完上面的 ✏️ 格，截一張**整個螢幕**的圖，畫面要同時看得到：

- **身份指紋**：左上的檔名、右上的 Google 頭貼、右下工作列的日期時間
- **這一題的輸出**：印出的 w、b、最後的 loss，以及 loss 曲線圖

看不到身份指紋，這一題不計分。

#### ✏️ 文字回答

**1. 一個 epoch 要搬幾盒？200 個 epoch 總共更新幾次參數？**

（在這裡寫）

**2. 把 opt.zero_grad() 那一行註解掉（前面加 #），先重跑「準備資料」再跑 ✏️ 格，loss 發生什麼事？為什麼？**

（在這裡寫）

> 先拍好截圖 ⑤ 再做第 2 題的實驗；做完把 `#` 拿掉，「準備資料」和 ✏️ 格都重跑一次，讓筆記本留下正確的結果。

## AI 使用紀錄

有用 AI 就寫下來，沒用就寫「沒有使用」。

| 用了哪個 AI | 問了什麼（大概就好） | 它哪裡幫到你 | 它哪裡說錯、你怎麼發現的 |
|---|---|---|---|
| （例：ChatGPT） | （例：requires_grad 是什麼意思） |  |  |

## 繳交前檢查

1. **重跑一次**：「執行階段 › 重新啟動工作階段並執行所有儲存格」，每一格都有輸出、沒有紅字。
2. **檔名**：`深度學習_W03延伸_學號_姓名.ipynb`（左上角點檔名就能改）。
3. **截圖**：5 張（①～⑤），每張都是整個螢幕，看得到身份指紋（左上的檔名、右上的 Google 頭貼、右下工作列的日期時間）和那一題的輸出。
4. **開放分享**：右上角「分享」→「一般存取權」改成「知道連結的任何人」→ 角色選「檢視者」→「複製連結」。
5. **自己檢查**：開一個無痕視窗貼上連結，不用登入就打得開才算成功。
6. **下載**：「檔案 › 下載 › 下載 .ipynb」。
7. **交到 Flipclass**：分享連結 1 個、.ipynb 檔 1 個、截圖 5 張（①～⑤），交到 Flipclass 作業區，期限照 Flipclass 規定。